In [ ]:
import boto3
import time
import logging
from datetime import datetime, date, timedelta, timezone
from typing import List, Optional, Tuple, Dict, Any
from decimal import Decimal

from botocore.exceptions import ClientError
from pyspark.sql import functions as F
from pyspark.sql import Row

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AWSCostExplorer")

logging.getLogger("boto3").setLevel(logging.WARNING)
logging.getLogger("botocore").setLevel(logging.WARNING)

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
overlap_days = int(dbutils.widgets.get("overlap_days") or "3")

if overlap_days < 2:
    logger.warning("overlap_days < 2; forcing to 2 for cost convergence best practice.")
    overlap_days = 2

audit_table = f"{catalog}.{schema}.dbspend360_audit_log"
target_table = f"{catalog}.{schema}.dbspend360_cloud_cost_explorer"

In [ ]:
# =======================================================
# AWS Cost Client
# =======================================================
class AWSCostClient:
    """Client for querying AWS Cost Explorer API for Databricks cluster costs.

    Uses Databricks service credentials to assume an IAM role with
    ce:GetCostAndUsage permissions. The Cost Explorer API endpoint
    is only available in us-east-1, regardless of where resources run.

    Queries costs grouped by the ClusterId tag that Databricks applies
    to EC2 instances, producing per-cluster daily costs aggregated
    across compute-related AWS services (EC2, EBS, S3, ELB, etc.).
    """

    CE_REGION = "us-east-1"

    DEFAULT_SERVICES = [
        "Amazon Elastic Compute Cloud - Compute",
        "Amazon Elastic Block Store",
        "Amazon Simple Storage Service",
        "Elastic Load Balancing",
        "AWS Data Transfer",
        "Amazon Virtual Private Cloud",
    ]

    MAX_CHUNK_DAYS = 30
    MAX_RETRIES = 5
    BASE_RETRY_DELAY = 5

    def __init__(self, service_credential_name: str = "dbspend-read-ce"):
        session = boto3.Session(
            botocore_session=dbutils.credentials.getServiceCredentialsProvider(
                service_credential_name
            ),
            region_name=self.CE_REGION,
        )
        self.client = session.client("ce")

    # -------- Public API --------

    def get_cluster_costs_daily(
        self,
        start_date: date,
        end_date: date,
        tag_key: str = "ClusterId",
        services: Optional[List[str]] = None,
        metric: str = "AmortizedCost",
    ):
        """Query CE for per-cluster daily costs, handling chunking and retries.

        Args:
            start_date: Inclusive start date.
            end_date: Inclusive end date.
            tag_key: EC2 tag key used by Databricks for cluster identification.
            services: AWS service names to include. Defaults to compute-related services.
            metric: CE cost metric (AmortizedCost recommended for RI/SP accuracy).

        Returns:
            Spark DataFrame with columns matching dbspend360_cloud_cost_explorer,
            or None if no cost data found.
        """
        if services is None:
            services = self.DEFAULT_SERVICES

        chunks = self._build_chunks(start_date, end_date)
        all_rows: List[Dict[str, Any]] = []

        for i, (chunk_start, chunk_end) in enumerate(chunks):
            logger.info(f"Querying CE chunk {i+1}/{len(chunks)}: {chunk_start} → {chunk_end}")
            rows = self._query_with_retries(
                chunk_start, chunk_end, tag_key, services, metric
            )
            all_rows.extend(rows)
            if len(chunks) > 1:
                time.sleep(1)

        if not all_rows:
            return None

        return self._rows_to_spark_df(all_rows)

    # -------- Chunking --------

    def _build_chunks(self, start: date, end: date) -> List[Tuple[date, date]]:
        chunks = []
        current = start
        while current <= end:
            chunk_end = min(current + timedelta(days=self.MAX_CHUNK_DAYS - 1), end)
            chunks.append((current, chunk_end))
            current = chunk_end + timedelta(days=1)
        return chunks

    # -------- CE params construction --------

    def _build_ce_params(
        self,
        start: date,
        end: date,
        tag_key: str,
        services: List[str],
        metric: str,
    ) -> dict:
        """Build GetCostAndUsage request parameters.

        Note: CE TimePeriod.End is exclusive, so we add 1 day to
        include the end_date in results.
        """
        return {
            "TimePeriod": {
                "Start": start.isoformat(),
                "End": (end + timedelta(days=1)).isoformat(),
            },
            "Granularity": "DAILY",
            "Metrics": [metric],
            "GroupBy": [{"Type": "TAG", "Key": tag_key}],
            "Filter": {"Dimensions": {"Key": "SERVICE", "Values": services}},
        }

    # -------- Retry logic --------

    def _query_with_retries(
        self,
        start: date,
        end: date,
        tag_key: str,
        services: List[str],
        metric: str,
    ) -> List[Dict[str, Any]]:
        """Execute a CE query with exponential-backoff retries.

        Handles AWS CE rate limiting (LimitExceededException) with
        progressively longer delays, and retries transient errors.
        """
        params = self._build_ce_params(start, end, tag_key, services, metric)
        last_exception = None

        for attempt in range(self.MAX_RETRIES):
            try:
                return self._execute_paginated_query(params, metric)
            except ClientError as e:
                last_exception = e
                error_code = e.response["Error"]["Code"]

                if error_code == "LimitExceededException":
                    wait = min(self.BASE_RETRY_DELAY * (2 ** attempt), 120)
                    logger.warning(
                        f"Rate limited (attempt {attempt + 1}/{self.MAX_RETRIES}), "
                        f"waiting {wait}s"
                    )
                    time.sleep(wait)
                elif attempt < self.MAX_RETRIES - 1:
                    wait = 2 ** attempt
                    logger.warning(
                        f"ClientError {error_code} (attempt {attempt + 1}), "
                        f"retrying in {wait}s: {e}"
                    )
                    time.sleep(wait)
                else:
                    raise
            except Exception as e:
                last_exception = e
                if attempt < self.MAX_RETRIES - 1:
                    wait = 2 ** attempt
                    logger.warning(
                        f"Unexpected error (attempt {attempt + 1}), "
                        f"retrying in {wait}s: {e}"
                    )
                    time.sleep(wait)
                else:
                    raise

        raise last_exception

    # -------- Pagination --------

    def _execute_paginated_query(
        self, params: dict, metric: str
    ) -> List[Dict[str, Any]]:
        """Execute a CE query, following pagination tokens to completion."""
        rows: List[Dict[str, Any]] = []
        request_params = params.copy()

        while True:
            response = self.client.get_cost_and_usage(**request_params)
            rows.extend(self._parse_response(response, metric))

            next_token = response.get("NextPageToken")
            if not next_token:
                break

            request_params["NextPageToken"] = next_token
            time.sleep(0.5)

        return rows

    # -------- Response parsing --------

    def _parse_response(
        self, response: dict, metric: str
    ) -> List[Dict[str, Any]]:
        """Parse a single CE response page into flat row dicts.

        CE tag group keys are formatted as "TagKey$TagValue" —
        we extract the value after the $ separator.
        """
        rows = []
        for time_block in response.get("ResultsByTime", []):
            period_date = time_block["TimePeriod"]["Start"]

            for group in time_block.get("Groups", []):
                keys = group.get("Keys", [])
                if not keys:
                    continue

                raw_tag = keys[0]
                cluster_id = raw_tag.split("$")[-1] if "$" in raw_tag else raw_tag

                metric_data = group["Metrics"].get(metric, {})
                amount = float(Decimal(metric_data.get("Amount", "0")))
                currency = metric_data.get("Unit", "USD")

                if amount == 0.0:
                    continue

                rows.append({
                    "cluster_id": cluster_id,
                    "cloud_cost": amount,
                    "currency": currency,
                    "cost_incurred_date": period_date,
                })

        return rows

    # -------- Spark conversion --------

    def _rows_to_spark_df(self, rows: List[Dict[str, Any]]):
        """Convert parsed rows to a Spark DataFrame with proper date types."""
        df = spark.createDataFrame(rows)
        return df.withColumn(
            "cost_incurred_date",
            F.to_date(F.col("cost_incurred_date"), "yyyy-MM-dd"),
        )

In [ ]:
# =======================================================
# APP
# =======================================================
class AWSCostReporterApp:
    """Orchestrates incremental AWS cost ingestion into the cloud cost table.

    Reads the audit log to determine the last successful run, queries
    AWS Cost Explorer for the incremental window (with overlap for
    idempotent MERGE), and upserts results into the target table.
    """

    def __init__(self):
        self.client = AWSCostClient()

    def run(self):
        start_dt, end_dt = self._get_date_window()

        logger.info(
            f"Querying AWS CE cost from {start_dt} to {end_dt} "
            f"(overlap_days={overlap_days})"
        )

        if start_dt > end_dt:
            message = (
                f"Invalid date window: start_dt={start_dt} > end_dt={end_dt}. "
                f"Check audit table and overlap_days={overlap_days}."
            )
            logger.error(message)
            self._log_run(start_dt, end_dt, "FAILED", 0, message)
            dbutils.notebook.exit("FAILED: Invalid date window.")

        spark_df = self.client.get_cluster_costs_daily(
            start_date=start_dt,
            end_date=end_dt,
        )

        if spark_df is None or spark_df.limit(1).count() == 0:
            logger.info("No AWS cost data returned for the requested range.")
            merged_row_count = 0
        else:
            inc_df = (
                spark_df
                .filter(
                    (F.col("cluster_id").isNotNull()) &
                    (F.col("cluster_id") != "")
                )
                .filter(F.col("cost_incurred_date").isNotNull())
            )

            if inc_df.limit(1).count() == 0:
                logger.info("No rows after filtering by cluster_id and cost_incurred_date.")
                merged_row_count = 0
            else:
                agg_df = (
                    inc_df
                    .groupBy("cluster_id", "currency", "cost_incurred_date")
                    .agg(F.sum("cloud_cost").alias("cloud_cost"))
                    .withColumn("created_at", F.current_timestamp())
                    .withColumn("updated_at", F.current_timestamp())
                )

                merged_row_count = agg_df.count()

                agg_df.createOrReplaceTempView("cloud_cost_inc")

                spark.sql(f"""
                MERGE INTO {target_table} AS t
                USING cloud_cost_inc AS s
                ON  t.cluster_id = s.cluster_id
                AND t.currency = s.currency
                AND t.cost_incurred_date = s.cost_incurred_date
                WHEN MATCHED THEN
                  UPDATE SET
                    t.cloud_cost = s.cloud_cost,
                    t.updated_at = current_timestamp()
                WHEN NOT MATCHED THEN
                  INSERT (cluster_id, cloud_cost, currency, cost_incurred_date, created_at, updated_at)
                  VALUES (s.cluster_id, s.cloud_cost, s.currency, s.cost_incurred_date,
                          current_timestamp(), current_timestamp())
                """)

        logger.info(
            f"Merged {merged_row_count} rows into {target_table} "
            f"for {start_dt} → {end_dt} (overlap_days={overlap_days})."
        )

        self._log_run(
            start_dt, end_dt, "SUCCESS", merged_row_count,
            f"overlap_days={overlap_days}"
        )

    def _get_date_window(self) -> Tuple[date, date]:
        """Determine the incremental date window from the audit log.

        On first run (no prior SUCCESS entries), defaults to 365 days back.
        On subsequent runs, uses the last successful end_date minus
        overlap_days for idempotent re-MERGE coverage.
        """
        wm = (
            spark.table(audit_table)
                 .filter("table_name = 'dbspend360_cloud_cost_explorer' AND status = 'SUCCESS'")
        )

        if wm.limit(1).count() == 0:
            last_end_date = datetime.now(timezone.utc).date() - timedelta(
                days=365 - overlap_days
            )
        else:
            last_end_date = wm.agg(F.max("end_date")).collect()[0][0]

        start_dt = last_end_date - timedelta(days=overlap_days - 1)
        end_dt = datetime.now(timezone.utc).date()
        return start_dt, end_dt

    def _log_run(self, start_dt, end_dt, status, row_count, message=""):
        """Append a run record to the audit log table."""
        run_log_df = spark.createDataFrame([
            Row(
                table_name="dbspend360_cloud_cost_explorer",
                start_date=start_dt,
                end_date=end_dt,
                status=status,
                row_count=int(row_count),
                message=message,
                created_at=datetime.now(timezone.utc),
            )
        ])
        run_log_df.write.mode("append").insertInto(audit_table)

In [ ]:
# =======================================================
# Execute
# =======================================================
app = AWSCostReporterApp()
app.run()